model tuning /
1-Veriyi bir kez eğitim ve nihai test olarak ayırırsın.
2-Eğitim verisini K-Fold ile birkaç kez farklı eğitim/doğrulama parçalarına bölersin.
3-Her turdaki doğrulama R² ve MSE sonuçlarını birlikte değerlendirirsin. Böylece performansın bölünmeye göre çok değişip değişmediğini görür, varsa farklı modelleri veya ayarları karşılaştırırsın.
4-Seçimini yaptıktan sonra modeli eğitim verisinin tamamıyla yeniden eğitirsin.
5-En son, daha önce dokunmadığın nihai test verisinde bir kez ölçüm yaparsın.

Bu işlem K-Fold çapraz doğrulamadır;K-Fold’da parçalar düzenli oluşturulur ve her satır bir kez doğrulama parçasında yer alır.

In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split,KFold
from sklearn.linear_model import LinearRegression
import sklearn.metrics as mt

In [16]:
dt=pd.read_csv("../16_basit_dogrusal_regresyon/data/advertising.csv")
veri=dt.copy()
veri

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9
...,...,...,...,...
195,38.2,3.7,13.8,7.6
196,94.2,4.9,8.1,14.0
197,177.0,9.3,6.4,14.8
198,283.6,42.0,66.2,25.5


In [35]:
y=veri["Sales"]
x=veri.drop(columns="Sales")
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
lr=LinearRegression()
model=lr.fit(x_train,y_train)


In [30]:
def skor(model,XX_train,XX_test,y_train,y_test):
    eğtim_tahmin=model.predict(XX_train)
    test_tahmin=model.predict(XX_test)

    r2_eğitim=mt.r2_score(y_train,eğtim_tahmin)
    r2_test=mt.r2_score(y_test,test_tahmin)

    mse_eğitim=mt.mean_squared_error(y_train,eğtim_tahmin)
    mse_test=mt.mean_squared_error(y_test,test_tahmin)

    return[r2_eğitim,r2_test,mse_eğitim,mse_test]


r2 ve mse hesaplamak için fonksiyon tanımlıyoruz

In [36]:
sonuç1=skor(model=lr,XX_train=x_train,XX_test=x_test,y_train=y_train,y_test=y_test)

print(f"eğitim r2={sonuç1[0]} eğitim mse={sonuç1[2]}")
print(f"test r2={sonuç1[1]} test mse={sonuç1[3]}")

eğitim r2=0.9001416005862131 eğitim mse=2.676142653782669
test r2=0.9059011844150826 test mse=2.9077569102710896


In [41]:
lr_cv=LinearRegression()
k=5
iterasyon=1
cv=KFold(n_splits=k)

for eğitim_index,test_index in cv.split(x):
    x_train,x_test=x.loc[eğitim_index],x.loc[test_index]
    y_train,y_test=y.loc[eğitim_index],y.loc[test_index]
    lr_cv.fit(x_train,y_train)

    sonuç2=skor(model=lr_cv,XX_train=x_train,XX_test=x_test,y_train=y_train,y_test=y_test)

    print(f"iterasyon:{iterasyon}")
    print(f"eğitim r2={sonuç2[0]} eğitim mse={sonuç2[2]}")
    print(f"test r2={sonuç2[1]} test mse={sonuç2[3]}")
    iterasyon+=1



iterasyon:1
eğitim r2=0.9076863645516066 eğitim mse=2.6387031967008094
test r2=0.8755626325832706 test mse=3.0560689723906513
iterasyon:2
eğitim r2=0.8943816583844933 eğitim mse=2.882791202477024
test r2=0.9317779087301498 test mse=2.0267606485198106
iterasyon:3
eğitim r2=0.8975973738365671 eğitim mse=2.9355376832423175
test r2=0.9215040344630709 test mse=1.851052120709308
iterasyon:4
eğitim r2=0.9142920378367592 eğitim mse=2.2557122003228898
test r2=0.8455458626911012 test mse=4.720392592253669
iterasyon:5
eğitim r2=0.9017116312831376 eğitim mse=2.7343971609584212
test r2=0.9024713241694589 test mse=2.6369407160816154


x verisinden eğitim data setini parçalama işlemini gerçekleştireceğiz,iterasyon 1 den başlayacakve biz kaç k değeri verdiysek ona göre parçalam işlemi yapacak ve her parça için eğitim,test r2,mse yapısı ortaya çıkaracak.

kfold ile tüm veri üzerinden regresyon sonucu alıyoruz,bu kısımda x_train kısmını y_test ve x_train şeklinde ayırmadan direkt x ve y değerlerini kullandık.2 farklı yolda kabul görmekte olup sayfanın başındaki yol daha iyi bir tercihtir.